In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_csv("used_cars_data.csv")

In [70]:
print(df.head())

   S.No.                              Name    Location  Year  \
0      0            Maruti Wagon R LXI CNG      Mumbai  2010   
1      1  Hyundai Creta 1.6 CRDi SX Option        Pune  2015   
2      2                      Honda Jazz V     Chennai  2011   
3      3                 Maruti Ertiga VDI     Chennai  2012   
4      4   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013   

   Kilometers_Driven Fuel_Type Transmission Owner_Type     Mileage   Engine  \
0              72000       CNG       Manual      First  26.6 km/kg   998 CC   
1              41000    Diesel       Manual      First  19.67 kmpl  1582 CC   
2              46000    Petrol       Manual      First   18.2 kmpl  1199 CC   
3              87000    Diesel       Manual      First  20.77 kmpl  1248 CC   
4              40670    Diesel    Automatic     Second   15.2 kmpl  1968 CC   

       Power  Seats  New_Price  Price  
0  58.16 bhp    5.0        NaN   1.75  
1  126.2 bhp    5.0        NaN  12.50  
2   88.7 bhp    5.0 

In [71]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7253 entries, 0 to 7252
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   S.No.              7253 non-null   int64  
 1   Name               7253 non-null   object 
 2   Location           7253 non-null   object 
 3   Year               7253 non-null   int64  
 4   Kilometers_Driven  7253 non-null   int64  
 5   Fuel_Type          7253 non-null   object 
 6   Transmission       7253 non-null   object 
 7   Owner_Type         7253 non-null   object 
 8   Mileage            7251 non-null   object 
 9   Engine             7207 non-null   object 
 10  Power              7207 non-null   object 
 11  Seats              7200 non-null   float64
 12  New_Price          1006 non-null   object 
 13  Price              6019 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 793.4+ KB
None


In [72]:
print(df.isnull().sum())

S.No.                   0
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 46
Power                  46
Seats                  53
New_Price            6247
Price                1234
dtype: int64


In [73]:
df.describe()

,S.No.,Year,Kilometers_Driven,Seats,Price
count,7253.000000,7253.000000,7.253000e+03,7200.000000,6019.000000
mean,3626.000000,2013.365366,5.869906e+04,5.279722,9.479468
std,2093.905084,3.254421,8.442772e+04,0.811660,11.187917
min,0.000000,1996.000000,1.710000e+02,0.000000,0.440000
25%,1813.000000,2011.000000,3.400000e+04,5.000000,3.500000
50%,3626.000000,2014.000000,5.341600e+04,5.000000,5.640000
75%,5439.000000,2016.000000,7.300000e+04,5.000000,9.950000
max,7252.000000,2019.000000,6.500000e+06,10.000000,160.000000


In [74]:
df.columns

Index(['S.No.', 'Name', 'Location', 'Year', 'Kilometers_Driven', 'Fuel_Type',
       'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats',
       'New_Price', 'Price'],
      dtype='object')

In [3]:
df["Engine_CC"] = df["Engine"].str.split().str[0].astype(float)

df["Engine_CC"] = df["Engine_CC"].fillna(df["Engine_CC"].mean())

In [76]:
print(df["Engine_CC"].dtype)

float64


In [77]:
df["Engine_CC"]

0        998.0
1       1582.0
2       1199.0
3       1248.0
4       1968.0
         ...  
7248    1598.0
7249    1197.0
7250    1461.0
7251    1197.0
7252    2148.0
Name: Engine_CC, Length: 7253, dtype: float64

In [4]:
df["Power_bhp"] = df["Power"].str.split().str[0]

df["Power_bhp"] = df["Power_bhp"].replace("null", np.nan)

df["Power_bhp"] = df["Power_bhp"].astype(float)

df["Power_bhp"] = df["Power_bhp"].fillna(df["Power_bhp"].mean())

In [79]:
print(df["Power_bhp"].dtype)

float64


In [80]:
df["Mileage"] = df["Mileage"].str.split().str[0].astype(float)

df["Mileage"] = df["Mileage"].fillna(df["Mileage"].mean())

In [81]:
df["Mileage"]

0       26.60
1       19.67
2       18.20
3       20.77
4       15.20
        ...  
7248    20.54
7249    17.21
7250    23.08
7251    17.20
7252    10.00
Name: Mileage, Length: 7253, dtype: float64

In [82]:
df["Seats"] = df["Seats"].fillna(5)

In [83]:
df["Seats"]

0       5.0
1       5.0
2       5.0
3       7.0
4       5.0
       ... 
7248    5.0
7249    5.0
7250    5.0
7251    5.0
7252    5.0
Name: Seats, Length: 7253, dtype: float64

In [84]:
df = df.dropna(subset=["Price"])

In [85]:
df

,S.No.,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price,Engine_CC,Power_bhp
0,0,Maruti Wagon R LXI CNG,Mumbai,2010,72000,CNG,Manual,First,26.60,998 CC,58.16 bhp,5.0,NaN,1.75,998.0,58.16
1,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67,1582 CC,126.2 bhp,5.0,NaN,12.50,1582.0,126.20
2,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.20,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50,1199.0,88.70
3,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77,1248 CC,88.76 bhp,7.0,NaN,6.00,1248.0,88.76
4,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.20,1968 CC,140.8 bhp,5.0,NaN,17.74,1968.0,140.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6014,6014,Maruti Swift VDI,Delhi,2014,27365,Diesel,Manual,First,28.40,1248 CC,74 bhp,5.0,7.88 Lakh,4.75,1248.0,74.00
6015,6015,Hyundai Xcent 1.1 CRDi S,Jaipur,2015,100000,Diesel,Manual,First,24.40,1120 CC,71 bhp,5.0,NaN,4.00,1120.0,71.00
6016,6016,Mahindra Xylo D4 BSIV,Jaipur,2012,55000,Diesel,Manual,Second,14.00,2498 CC,112 bhp,8.0,NaN,2.90,2498.0,112.00
6017,6017,Maruti Wagon R VXI,Kolkata,2013,46000,Petrol,Manual,First,18.90,998 CC,67.1 bhp,5.0,NaN,2.65,998.0,67.10


In [86]:
print(df.shape)
print(df.columns)
print(df.dtypes)

(6019, 16)
Index(['S.No.', 'Name', 'Location', 'Year', 'Kilometers_Driven', 'Fuel_Type',
       'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats',
       'New_Price', 'Price', 'Engine_CC', 'Power_bhp'],
      dtype='object')
S.No.                  int64
Name                  object
Location              object
Year                   int64
Kilometers_Driven      int64
Fuel_Type             object
Transmission          object
Owner_Type            object
Mileage              float64
Engine                object
Power                 object
Seats                float64
New_Price             object
Price                float64
Engine_CC            float64
Power_bhp            float64
dtype: object


In [87]:
df.drop(["New_Price"], axis=1, inplace=True)

In [88]:
df.drop(["Engine","Power"], axis=1, inplace=True)

In [89]:
df.drop(["S.No.","Name"], axis=1, inplace=True)

In [5]:
df = pd.get_dummies(
    df,
    columns=[
        "Location",
        "Fuel_Type",
        "Transmission",
        "Owner_Type"
    ],
    drop_first=True
)

In [6]:
df["Price_Category"] = pd.qcut(
    df["Price"],
    q=5,
    labels=[1,2,3,4,5]
)

In [7]:
df["Price_Category"]

0         1
1         4
2         2
3         3
4         5
       ... 
7248    NaN
7249    NaN
7250    NaN
7251    NaN
7252    NaN
Name: Price_Category, Length: 7253, dtype: category
Categories (5, int64): [1 < 2 < 3 < 4 < 5]

In [92]:
split = StratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

for train_index, test_index in split.split(df, df["Price_Category"]):

    train_set = df.loc[train_index]

    test_set = df.loc[test_index]

In [93]:
train_set = train_set.drop("Price_Category",axis=1)

test_set = test_set.drop("Price_Category",axis=1)

In [94]:
X_train = train_set.drop("Price",axis=1)

y_train = train_set["Price"]

X_test = test_set.drop("Price",axis=1)

y_test = test_set["Price"]

In [95]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [96]:
model = LinearRegression()

model.fit(X_train,y_train)

LinearRegression()

In [97]:
y_pred = model.predict(X_test)

In [101]:
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print("R2 Score :", r2)
print("MSE :", mse)
print("RMSE :", rmse)

R2 Score : 0.7276571144860087
MSE : 30.185078711233295
RMSE : 5.494094894633082
